# Elliptic Bitcoin Transactions: GraphSAGE Training, Stacking and Ablation

Notebook 4 of 7. With the settings tuned in notebook 3, it trains 15 GraphSAGE models (5 temporal folds x 3 seeds), produces leak-free out-of-fold logits for the training period and 15-model test logits, turns them into stacking features for LightGBM, measures what focal loss and balanced sampling actually contribute, and exports the weights for the serving app.

This notebook writes test-period scores but computes no test metric; the test set is evaluated once, in notebook 5.

## Setup

The same model, sampling, loss and training functions as notebook 3, plus the stacking-feature functions from `app/graph.py` and a NumPy port of the forward pass from `app/sage.py`, used to check the serving implementation against PyTorch here on Kaggle. `SMOKE` keeps the full graph with one seed and five epochs; it is off in this version.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", "torch_geometric==2.8.0.post1"], check=True, capture_output=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch_geometric
from cycler import cycler
from scipy import sparse
from sklearn.metrics import average_precision_score, precision_recall_curve
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import remove_self_loops, to_undirected

SMOKE = False
SEED = 42
THREADS = 4
TRAIN_LAST = 34
INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working")
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "text.color": INK, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2, "legend.frameon": False,
    "font.family": "sans-serif", "axes.prop_cycle": cycler(color=SERIES), "axes.titlesize": 11,
})
torch.set_num_threads(THREADS)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_default_dtype(torch.float64)
START = time.time()

STRUCTURAL = ["in_degree", "out_degree", "pagerank", "core_number", "clustering", "avg_neighbor_degree", "two_hop_size"]
STACKING = ["gnn_logit", "gnn_in_mean", "gnn_in_max", "gnn_out_mean", "gnn_out_max"]
LOG1P = ("in_degree", "out_degree", "core_number", "avg_neighbor_degree", "two_hop_size")
LOG = ("pagerank",)
CONFIGS = [("bce_natural", "bce", False), ("bce_balanced", "bce", True), ("focal_natural", "focal", False), ("focal_balanced", "focal", True)]


def gnn_matrix(raw, structural):
    values = np.array(structural, dtype=float)
    for name in LOG1P:
        i = STRUCTURAL.index(name)
        values[:, i] = np.log1p(values[:, i])
    for name in LOG:
        i = STRUCTURAL.index(name)
        values[:, i] = np.log(values[:, i])
    return np.hstack([np.asarray(raw, dtype=float), values])


class SAGE(torch.nn.Module):
    def __init__(self, d_in, hidden, layers, dropout):
        super().__init__()
        self.inp = torch.nn.Linear(d_in, hidden)
        self.convs = torch.nn.ModuleList([SAGEConv(hidden, hidden, aggr="mean") for _ in range(layers)])
        self.head = torch.nn.Linear(hidden * (layers + 1), 1)
        self.dropout = dropout

    def forward(self, x, edge_indices):
        h = F.dropout(F.relu(self.inp(x)), self.dropout, self.training)
        hidden = [h]
        for conv, edge_index in zip(self.convs, edge_indices):
            h = F.dropout(F.relu(conv(h, edge_index)), self.dropout, self.training)
            hidden.append(h)
        return self.head(torch.cat(hidden, dim=1)).squeeze(-1)


def sample_edges(edge_index, num_nodes, k, gen):
    dst = edge_index[1]
    key = dst.to(torch.float64) + torch.rand(dst.numel(), generator=gen, dtype=torch.float64)
    order = torch.argsort(key, stable=True)
    sorted_dst = dst[order]
    counts = torch.bincount(dst, minlength=num_nodes)
    starts = torch.cumsum(counts, 0) - counts
    rank = torch.arange(dst.numel()) - starts[sorted_dst]
    return edge_index[:, order[rank < k]]


def epoch_seeds(positives, negatives, ratio, gen):
    if ratio is None:
        return torch.cat([positives, negatives])
    take = min(len(negatives), int(ratio) * len(positives))
    pick = negatives[torch.randperm(len(negatives), generator=gen)[:take]]
    return torch.cat([positives, pick])


def focal_loss(logits, target, alpha, gamma):
    ce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    modulating = torch.exp(gamma * F.logsigmoid(logits * (1 - 2 * target)))
    a_t = alpha * target + (1 - alpha) * (1 - target)
    return (a_t * modulating * ce).mean()


def predict(model, x, edge):
    model.eval()
    with torch.no_grad():
        return model(x, [edge] * len(model.convs)).numpy().astype(np.float64)


def fit(params, fold, seed, epochs, checkpoints=(), balance=True, loss="focal"):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    model = SAGE(X.shape[1], params["hidden"], params["layers"], params["dropout"])
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    in_fit = (FOLD != fold) & LABELLED
    positives = torch.tensor(np.flatnonzero(in_fit & (TRUE == 1)))
    negatives = torch.tensor(np.flatnonzero(in_fit & (TRUE == 0)))
    held = np.flatnonzero(FOLD == fold)
    history = {}
    for epoch in range(1, epochs + 1):
        model.train()
        seeds = epoch_seeds(positives, negatives, params["ratio"] if balance else None, gen)
        layer_edges = [sample_edges(EDGE, N, params["fanout"], gen) for _ in range(params["layers"])]
        out = model(X, layer_edges)[seeds]
        if loss == "focal":
            value = focal_loss(out, Y[seeds], params["alpha"], params["gamma"])
        else:
            value = F.binary_cross_entropy_with_logits(out, Y[seeds])
        opt.zero_grad()
        value.backward()
        opt.step()
        if epoch in checkpoints:
            history[epoch] = predict(model, X, EDGE)[held]
    return model, history


def directed(n, src, dst):
    src = np.asarray(src, dtype=np.int64)
    dst = np.asarray(dst, dtype=np.int64)
    keep = src != dst
    matrix = sparse.csr_matrix((np.ones(int(keep.sum())), (src[keep], dst[keep])), shape=(n, n))
    matrix.sum_duplicates()
    matrix.data[:] = 1.0
    return matrix


def segment_mean(matrix, values):
    counts = np.diff(matrix.indptr).astype(float)
    sums = np.asarray(matrix @ values, dtype=float)
    out = np.full(sums.shape, np.nan)
    has = counts > 0
    out[has] = sums[has] / counts[has, None]
    return out


def segment_max(matrix, values):
    out = np.full((matrix.shape[0], values.shape[1]), np.nan)
    has = np.diff(matrix.indptr) > 0
    if has.any():
        out[has] = np.maximum.reduceat(values[matrix.indices], matrix.indptr[:-1][has], axis=0)
    return out


def stacking_features(n, src, dst, logits):
    forward = directed(n, src, dst)
    backward = forward.T.tocsr()
    values = np.asarray(logits, dtype=float).reshape(n, 1)
    return np.hstack([
        values,
        segment_mean(backward, values),
        segment_max(backward, values),
        segment_mean(forward, values),
        segment_max(forward, values),
    ])


def symmetric(n, src, dst):
    src = np.asarray(src, dtype=np.int64)
    dst = np.asarray(dst, dtype=np.int64)
    rows = np.concatenate([dst, src])
    cols = np.concatenate([src, dst])
    keep = rows != cols
    adj = sparse.csr_matrix((np.ones(int(keep.sum())), (rows[keep], cols[keep])), shape=(n, n))
    adj.sum_duplicates()
    adj.data[:] = 1.0
    return adj


def normalise(adj):
    degree = np.diff(adj.indptr).astype(float)
    scale = np.zeros(adj.shape[0])
    scale[degree > 0] = 1.0 / degree[degree > 0]
    return (sparse.diags(scale) @ adj).tocsr()


def numpy_forward(params, x, adj, layers):
    h = np.maximum(x @ params["inp.weight"].T + params["inp.bias"], 0.0)
    hidden = [h]
    for layer in range(layers):
        prefix = f"convs.{layer}."
        h = np.maximum((adj @ h) @ params[prefix + "lin_l.weight"].T + params[prefix + "lin_l.bias"] + h @ params[prefix + "lin_r.weight"].T, 0.0)
        hidden.append(h)
    return (np.hstack(hidden) @ params["head.weight"].T + params["head.bias"]).ravel()


def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-np.asarray(values, dtype=float)))


def best_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    f1 = 2 * precision[:-1] * recall[:-1] / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    i = int(np.nanargmax(f1))
    return float(thresholds[i]), float(f1[i])


def find_file(name):
    hits = sorted(INPUT.rglob(name))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def elapsed():
    return f"elapsed {(time.time() - START) / 60:.1f} minutes"


best = json.loads(find_file("best_params.json").read_text(encoding="utf-8"))
EPOCHS = 5 if SMOKE else int(best["epochs"])
SEEDS = [42] if SMOKE else [42, 43, 44]
print(f"smoke={SMOKE}, epochs={EPOCHS}, seeds={SEEDS}")
print(f"torch {torch.__version__}, torch_geometric {torch_geometric.__version__}, numpy {np.__version__}")
print(json.dumps(best, indent=2))

**Conclusion.** The full run trains every fold model for the tuned 75 epochs with three seeds (42, 43, 44), in float64, using notebook 3's winning settings: 2 layers, hidden size 64, focal alpha 0.654 and gamma 1.645, a 1:4 seed ratio and a fan-out of 25.

## Build the training and test graphs

Aim: assemble the training graph exactly as in notebook 3 (same inputs, same standardisation) and a separate graph of steps 35 to 49 standardised with the training statistics. Since no edge crosses time steps, the test graph is complete on its own.

In [ ]:
feat = pd.read_parquet(find_file("features.parquet"))
split = pd.read_parquet(find_file("split.parquet"))
edges = pd.read_parquet(find_file("edges.parquet"))
meta = json.loads(find_file("feature_meta.json").read_text(encoding="utf-8"))
RAW = meta["raw"]
train = feat[feat.time_step <= TRAIN_LAST].merge(split[["txId", "temporal_fold"]], on="txId").reset_index(drop=True)
x_np = gnn_matrix(train[RAW].to_numpy(np.float64), train[STRUCTURAL].to_numpy(np.float64))
MEAN = x_np.mean(axis=0)
STD = x_np.std(axis=0)
STD[STD == 0] = 1.0
X = torch.tensor((x_np - MEAN) / STD, dtype=torch.float64)
LABEL = train.label.to_numpy()
LABELLED = LABEL >= 0
TRUE = (LABEL == 1).astype(int)
Y = torch.tensor(TRUE, dtype=torch.float64)
FOLD = train.temporal_fold.to_numpy()
N = len(train)


def graph_edges(frame):
    index = pd.Series(np.arange(len(frame)), index=frame.txId.to_numpy())
    inside = edges.txId1.isin(index.index) & edges.txId2.isin(index.index)
    pairs = torch.tensor(np.vstack([index.loc[edges.txId1[inside]].to_numpy(), index.loc[edges.txId2[inside]].to_numpy()]), dtype=torch.long)
    pairs, _ = remove_self_loops(pairs)
    return to_undirected(pairs, num_nodes=len(frame))


EDGE = graph_edges(train)
test = feat[feat.time_step > TRAIN_LAST].reset_index(drop=True)
x_test = (gnn_matrix(test[RAW].to_numpy(np.float64), test[STRUCTURAL].to_numpy(np.float64)) - MEAN) / STD
X_TEST = torch.tensor(x_test, dtype=torch.float64)
EDGE_TEST = graph_edges(test)
print(f"training graph: {N:,} nodes, {EDGE.shape[1]:,} symmetrised edges")
print(f"test graph: {len(test):,} nodes, {EDGE_TEST.shape[1]:,} symmetrised edges")

**Conclusion.** The training graph is identical to notebook 3's (136,265 nodes, 313,686 symmetrised edges). The test graph holds all 67,504 transactions of steps 35 to 49 with 155,024 symmetrised edges, standardised with training-period statistics only, so nothing about the test period shapes the inputs.

## Fifteen fold models

Aim: for each temporal fold and each of three seeds, train on the other four folds for the tuned epoch count, score every node of the held-out fold (labelled or not) and every test node. A training-period node's out-of-fold logit is the mean of its fold's three models, so no model ever scores a node whose neighbourhood it trained on; a test node's logit is the mean of all 15.

In [ ]:
oof = np.zeros(N)
test_logit = np.zeros(len(test))
states = []
first_test = None
for fold in range(5):
    held = np.flatnonzero(FOLD == fold)
    fold_pred = np.zeros(len(held))
    for seed in SEEDS:
        model, _ = fit(best, fold, seed, EPOCHS)
        fold_pred += predict(model, X, EDGE)[held] / len(SEEDS)
        out_test = predict(model, X_TEST, EDGE_TEST)
        test_logit += out_test / (5 * len(SEEDS))
        if first_test is None:
            first_test = out_test
        states.append({k: v.detach().numpy().copy() for k, v in model.state_dict().items()})
    oof[held] = fold_pred
    lab = LABELLED[held]
    print(f"fold {fold}: out-of-fold pr_auc {average_precision_score(TRUE[held][lab], fold_pred[lab]):.4f}")
    print(f"fold {fold} {elapsed()}")
p_oof = sigmoid(oof[LABELLED])
threshold, f1 = best_threshold(TRUE[LABELLED], p_oof)
print(f"pooled out-of-fold pr_auc {average_precision_score(TRUE[LABELLED], p_oof):.4f}, best f1 {f1:.4f} at threshold {threshold:.4f}")
print(f"models trained: {len(states)}")

**Conclusion.** Fifteen models trained in 41 minutes. Pooled across the training period, the out-of-fold PR-AUC is 0.9453 and the best illicit F1 is 0.896 at a threshold of 0.647. The per-fold scores show the drift problem in miniature: folds 1 to 4 score 0.946 to 0.978, but fold 0 (steps 1 to 4, only 76 illicit transactions) scores 0.144, because models trained on later steps do not recognise the early steps' illicit patterns. The test period lies on the other side of the same kind of shift.

## Check the NumPy port

Aim: confirm that the serving app's NumPy forward pass reproduces PyTorch Geometric's logits for the first model on the full test graph, so the live API scores transactions exactly as trained. Both run in float64, so they should agree to rounding error.

In [ ]:
params = {k: v.astype(np.float64) for k, v in states[0].items()}
expected = {"inp.weight", "inp.bias", "head.weight", "head.bias"} | {f"convs.{l}.{p}" for l in range(best["layers"]) for p in ("lin_l.weight", "lin_l.bias", "lin_r.weight")}
print(f"state dict keys match the artifact contract: {set(params) == expected}")
adj = normalise(symmetric(len(test), EDGE_TEST[0].numpy(), EDGE_TEST[1].numpy()))
numpy_logits = numpy_forward(params, x_test, adj, best["layers"])
print(f"numpy port matches torch within 1e-9 on every test logit: {bool(np.abs(numpy_logits - first_test).max() < 1e-9)}")

**Conclusion.** The state dict holds exactly the keys the artifact contract names, and the NumPy forward pass the serving app uses reproduces PyTorch Geometric's float64 logits for the first model on every one of the 67,504 test transactions to within 1e-9, so the live API scores transactions exactly as trained.

## Stacking features

Aim: turn the GraphSAGE logits into five features for LightGBM: each node's own logit and the mean and max logit over its inputs and over its outputs. Logits are rounded to float32 first, the precision the serving store keeps, so the app recomputes these features exactly.

In [ ]:
pos_all = pd.Series(np.arange(len(feat)), index=feat.txId.to_numpy())
logit_all = np.full(len(feat), np.nan)
logit_all[pos_all.loc[train.txId].to_numpy()] = oof
logit_all[pos_all.loc[test.txId].to_numpy()] = test_logit
print(f"nodes without a logit: {int(np.isnan(logit_all).sum())}")
src = pos_all.loc[edges.txId1].to_numpy()
dst = pos_all.loc[edges.txId2].to_numpy()
stack = stacking_features(len(feat), src, dst, logit_all.astype(np.float32)).astype(np.float32)
scores = pd.DataFrame(stack, columns=STACKING)
scores.insert(0, "set", np.where(feat.time_step.to_numpy() <= TRAIN_LAST, "oof", "test"))
scores.insert(0, "time_step", feat.time_step.to_numpy())
scores.insert(0, "txId", feat.txId.to_numpy())
scores.to_parquet(OUT / "gnn_scores.parquet", index=False)
print(scores.groupby("set")[STACKING].describe().T.round(4).to_string())

**Conclusion.** Every node now has a GraphSAGE logit: out-of-fold for steps 1 to 34 and the 15-model mean for steps 35 to 49. The two sets have similar centres (median logit -3.57 out-of-fold against -3.85 on test), so stacking features built from them are on a comparable scale. Input aggregates exist for 101,527 training-period and 46,920 test-period nodes, output aggregates for 112,247 and 54,098; the rest are missing, which LightGBM handles natively. A few extreme negative logits (down to -365) are harmless to tree models, which only use their order.

## Imbalance ablation

Aim: measure, rather than assert, what the two imbalance tools add. The tuned architecture is retrained (one seed, five folds) four ways: plain cross-entropy or the tuned focal loss, each with class-balanced seed sampling on or off (off means every labelled training node is a seed at its natural ratio; neighbourhood fan-out stays on in all four). Out-of-fold results are printed here; test scores are written for notebook 5.

In [ ]:
ABL_EPOCHS = 3 if SMOKE else EPOCHS
abl_frame = pd.DataFrame({"txId": feat.txId.to_numpy(), "set": scores["set"].to_numpy()})
ablation = {}
for name, loss, balance in CONFIGS:
    o = np.zeros(N)
    t = np.zeros(len(test))
    for fold in range(5):
        held = np.flatnonzero(FOLD == fold)
        model, _ = fit(best, fold, SEED, ABL_EPOCHS, balance=balance, loss=loss)
        o[held] = predict(model, X, EDGE)[held]
        t += predict(model, X_TEST, EDGE_TEST) / 5
    column = np.full(len(feat), np.nan)
    column[pos_all.loc[train.txId].to_numpy()] = o
    column[pos_all.loc[test.txId].to_numpy()] = t
    abl_frame[name] = column.astype(np.float32)
    p = sigmoid(o[LABELLED])
    thr, f1 = best_threshold(TRUE[LABELLED], p)
    ablation[name] = {"pr_auc": average_precision_score(TRUE[LABELLED], p), "f1": f1}
    print(f"{name}: out-of-fold pr_auc {ablation[name]['pr_auc']:.4f}, f1 {f1:.4f} at threshold {thr:.4f}")
    print(f"{name} {elapsed()}")
abl_frame.to_parquet(OUT / "ablation_scores.parquet", index=False)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
names = [c[0] for c in CONFIGS]
values = [ablation[n]["pr_auc"] for n in names]
ax.barh(names, values, color=SERIES[0], height=0.6)
for y, value in enumerate(values):
    ax.text(value, y, f" {value:.3f}", va="center", color=INK_2, fontsize=9)
ax.set(xlabel="pooled out-of-fold PR-AUC", title="Imbalance ablation (tuned GraphSAGE, one seed)")
save(fig, "ablation_oof")

**Conclusion.** The ablation separates the two imbalance tools. Used alone, each adds little over plain cross-entropy at the natural class ratio (pooled out-of-fold PR-AUC 0.928): balanced seed sampling reaches 0.930 and focal loss 0.931. Together they reach 0.939, and F1 rises from 0.874 to 0.885. So the combination of focal loss and graph-aware sampling is what helps, by about 0.011 PR-AUC and 0.011 F1 on the training period, while neither alone is worth much. Notebook 5 checks whether the same ordering holds on the test period.

## Export the weights

Aim: save the 15 models with the standardisation statistics in the format the serving app reads (`graphsage.npz`), in float64 so the app reproduces the trained logits exactly, and record their size.

In [ ]:
arrays = {"mean": MEAN, "std": STD, "layers": np.array(best["layers"]), "hidden": np.array(best["hidden"]), "n_models": np.array(len(states))}
for k, state in enumerate(states):
    for name, value in state.items():
        arrays[f"m{k}.{name}"] = value.astype(np.float64)
np.savez(OUT / "graphsage.npz", **arrays)
count = sum(v.size for v in states[0].values())
print(f"parameters per model: {count:,}")
print(f"graphsage.npz: {(OUT / 'graphsage.npz').stat().st_size / 1e6:.1f} MB")
print(elapsed())

**Conclusion.** The 15 models have 27,777 parameters each and the whole export, stored in float64 with the standardisation statistics, is 3.4 MB, small enough for the serving container to load in full and average all 15 at request time.